# GF-MUL Decoder SASCA - Gaussian posterior calibrated to paper Table 2 accuracies

이 노트북은 논문 Table 2의 HW template accuracy를 **Gaussian HW leakage model**로 맞춘 뒤 decoder SASCA를 30회 반복 평가한다.

핵심 모델:

\[
x = \alpha \cdot h + \beta + N(0,\sigma_{raw}^2), \quad h=HW(v)
\]

정확도는 \(\alpha\)와 \(\sigma_{raw}\) 각각이 아니라

\[
\rho = \frac{|\alpha|}{\sigma_{raw}}, \qquad \sigma_{eff}=\frac{1}{\rho}=\frac{\sigma_{raw}}{|\alpha|}
\]

에 의해 결정된다. 따라서 Table 2 accuracy를 먼저 \(\rho\)로 변환하고, 사용자가 설정한 \(\alpha\)에 따라 \(\sigma_{raw}=|\alpha|/\rho\)를 계산한다.

기본값:
- `op1_hw` accuracy = 0.3035
- `out_hw` accuracy = 0.5178
- `N_ATTACKS = 30`

`ALPHA_OP1`, `ALPHA_OUT`을 바꾸면 같은 accuracy를 유지하기 위해 raw sigma가 어떻게 바뀌는지 확인할 수 있다.


In [1]:
import numpy as np
from math import erf, sqrt
from scalib.attacks import FactorGraph, BPState

# ----------------------------
# User config
# ----------------------------
SEED = 1234
N_ATTACKS = 30
TOPK = 32

# 논문 Table 2 HW template accuracy
ACC_OP1_HW = 0.3035
ACC_OUT_HW = 0.5178

# 사용자가 조정할 수 있는 raw leakage scale
# alpha가 바뀌면 같은 accuracy를 맞추기 위해 sigma_raw도 같이 바뀐다.
ALPHA_OP1 = 1.0
ALPHA_OUT = 4.0
BETA_OP1 = 0.0
BETA_OUT = 0.0

rng = np.random.default_rng(SEED)


## 수식적 근거

Gaussian leakage를

\[
x=\alpha h+\beta+\epsilon,\quad \epsilon\sim N(0,\sigma_{raw}^{2})
\]

라고 두자. 여기서 \(h\in\{0,\dots,8\}\)는 HW class다.

가장 가까운 HW 평균값 \(\alpha k+\beta\)를 고르는 classifier의 decision boundary는 인접한 두 class 사이의 중점이다. 내부 class \(1\le h\le 7\)에서 정답이 되려면

\[
\alpha(h-\tfrac12)+\beta \le x < \alpha(h+\tfrac12)+\beta
\]

이어야 한다. 따라서

\[
-\frac{|\alpha|}{2} \le \epsilon < \frac{|\alpha|}{2}
\]

이고,

\[
P(\text{correct}\mid h)=
2\Phi\left(\frac{|\alpha|}{2\sigma_{raw}}\right)-1
=
2\Phi\left(\frac{\rho}{2}\right)-1
\]

이다. 단,

\[
\rho=\frac{|\alpha|}{\sigma_{raw}}.
\]

끝점 \(h=0,8\)은 한쪽 boundary만 있으므로

\[
P(\text{correct}\mid h=0\text{ or }8)=\Phi(\rho/2)
\]

로 계산한다. 최종 accuracy는 byte HW 분포

\[
P(HW=h)=\binom{8}{h}/256
\]

로 가중 평균한다. 이 노트북은 target accuracy에서 \(\rho\)를 먼저 찾고, 사용자가 지정한 \(\alpha\)에 대해

\[
\sigma_{raw}=\frac{|\alpha|}{\rho}
\]

를 계산한다. 그래서 \(\alpha\)를 바꿔도 target accuracy가 유지된다.


In [2]:
# ----------------------------
# Gaussian model calibrated from top-1 accuracy
# ----------------------------
HW_WEIGHTS = np.array([1, 8, 28, 56, 70, 56, 28, 8, 1], dtype=np.float64) / 256.0

def Phi(x: float) -> float:
    return 0.5 * (1.0 + erf(float(x) / sqrt(2.0)))

def hw_top1_accuracy_from_rho(rho: float) -> float:
    """
    Nearest-HW classifier accuracy under:
        x = alpha*h + beta + N(0, sigma_raw^2)
    where rho = |alpha|/sigma_raw.

    For internal HW classes 1..7:
        correct iff |noise| < |alpha|/2
    For endpoints 0 and 8:
        one-sided interval.
    HW distribution is byte HW ~ Binomial(8, 1/2).
    """
    rho = float(rho)
    acc = 0.0
    for h, w in enumerate(HW_WEIGHTS):
        if h == 0 or h == 8:
            p_correct = Phi(rho / 2.0)
        else:
            p_correct = 2.0 * Phi(rho / 2.0) - 1.0
        acc += w * p_correct
    return acc

def find_rho_for_accuracy(target_acc: float, lo: float = 1e-9, hi: float = 100.0, iters: int = 100) -> float:
    """Solve A(rho)=target_acc by binary search."""
    target_acc = float(target_acc)
    if not (hw_top1_accuracy_from_rho(lo) <= target_acc <= hw_top1_accuracy_from_rho(hi)):
        raise ValueError(f"target_acc={target_acc} is outside achievable range")
    for _ in range(iters):
        mid = (lo + hi) / 2.0
        if hw_top1_accuracy_from_rho(mid) < target_acc:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2.0

RHO_OP1 = find_rho_for_accuracy(ACC_OP1_HW)
RHO_OUT = find_rho_for_accuracy(ACC_OUT_HW)

SIGMA_EFF_OP1 = 1.0 / RHO_OP1
SIGMA_EFF_OUT = 1.0 / RHO_OUT

def sigma_raw_from_alpha(alpha: float, rho: float) -> float:
    if abs(alpha) == 0:
        raise ValueError("alpha must be nonzero")
    return abs(float(alpha)) / float(rho)

SIGMA_RAW_OP1 = sigma_raw_from_alpha(ALPHA_OP1, RHO_OP1)
SIGMA_RAW_OUT = sigma_raw_from_alpha(ALPHA_OUT, RHO_OUT)

def gaussian_hw_posterior(true_hw: int, alpha: float, beta: float, sigma_raw: float, rng):
    """
    Generate one noisy leakage x and convert it into posterior over HW classes 0..8.

        x = alpha*true_hw + beta + N(0, sigma_raw^2)
        P(k | x) ∝ exp(-(x - (alpha*k + beta))^2 / (2*sigma_raw^2))

    This is the Gaussian-like posterior used as SASCA input.
    """
    true_hw = int(true_hw)
    alpha = float(alpha)
    beta = float(beta)
    sigma_raw = float(sigma_raw)
    x = alpha * true_hw + beta + rng.normal(0.0, sigma_raw)

    hws = np.arange(9, dtype=np.float64)
    means = alpha * hws + beta
    logp = -0.5 * ((x - means) / sigma_raw) ** 2
    p = np.exp(logp - np.max(logp))
    return p / p.sum()

print("[Gaussian calibration from Table 2 accuracy]")
print(f"op1 target acc = {ACC_OP1_HW:.4f} -> rho={RHO_OP1:.6f}, sigma_eff={SIGMA_EFF_OP1:.6f}")
print(f"out target acc = {ACC_OUT_HW:.4f} -> rho={RHO_OUT:.6f}, sigma_eff={SIGMA_EFF_OUT:.6f}")
print()
print("[Current alpha -> raw sigma]")
print(f"ALPHA_OP1={ALPHA_OP1} -> SIGMA_RAW_OP1={SIGMA_RAW_OP1:.6f}")
print(f"ALPHA_OUT={ALPHA_OUT} -> SIGMA_RAW_OUT={SIGMA_RAW_OUT:.6f}")


[Gaussian calibration from Table 2 accuracy]
op1 target acc = 0.3035 -> rho=0.772716, sigma_eff=1.294136
out target acc = 0.5178 -> rho=1.399476, sigma_eff=0.714553

[Current alpha -> raw sigma]
ALPHA_OP1=1.0 -> SIGMA_RAW_OP1=1.294136
ALPHA_OUT=4.0 -> SIGMA_RAW_OUT=2.858213


In [3]:
# ----------------------------
# Sigma values for different alpha choices
# ----------------------------
ALPHA_GRID = [0.25, 0.5, 1.0, 2.0, 4.0]

print("[sigma_raw as alpha changes while accuracy is fixed]")
print("alpha | sigma_raw(op1 acc=0.3035) | sigma_raw(out acc=0.5178)")
for a in ALPHA_GRID:
    sig_op1 = sigma_raw_from_alpha(a, RHO_OP1)
    sig_out = sigma_raw_from_alpha(a, RHO_OUT)
    print(f"{a:5.2f} | {sig_op1:25.6f} | {sig_out:25.6f}")

def empirical_hw_accuracy(acc_name, alpha, beta, sigma_raw, n=200000, seed=7):
    rng_local = np.random.default_rng(seed)
    true_hw = rng_local.choice(np.arange(9), size=n, p=HW_WEIGHTS)
    pred = np.empty(n, dtype=np.int16)
    for i, h in enumerate(true_hw):
        p = gaussian_hw_posterior(int(h), alpha, beta, sigma_raw, rng_local)
        pred[i] = int(np.argmax(p))
    measured = float(np.mean(pred == true_hw))
    print(f"{acc_name}: empirical top-1 acc ≈ {measured:.5f}")

print("\n[empirical sanity check]")
empirical_hw_accuracy("op1", ALPHA_OP1, BETA_OP1, SIGMA_RAW_OP1, n=50000, seed=11)
empirical_hw_accuracy("out", ALPHA_OUT, BETA_OUT, SIGMA_RAW_OUT, n=50000, seed=12)


[sigma_raw as alpha changes while accuracy is fixed]
alpha | sigma_raw(op1 acc=0.3035) | sigma_raw(out acc=0.5178)
 0.25 |                  0.323534 |                  0.178638
 0.50 |                  0.647068 |                  0.357277
 1.00 |                  1.294136 |                  0.714553
 2.00 |                  2.588271 |                  1.429106
 4.00 |                  5.176543 |                  2.858213

[empirical sanity check]
op1: empirical top-1 acc ≈ 0.30628
out: empirical top-1 acc ≈ 0.51776


In [4]:
# ----------------------------
# Common helpers
# ----------------------------
def hw8_scalar(x: int) -> int:
    return bin(x & 0xFF).count("1")

def gf_mul_ref_python(a: int, b: int) -> int:
    a &= 0xFF
    b &= 0xFF
    p = 0
    for _ in range(8):
        if b & 1:
            p ^= a
        hi = a & 0x80
        a = (a << 1) & 0xFF
        if hi:
            a ^= 0x1D
        b >>= 1
    return p & 0xFF

def posterior_rank(post: np.ndarray, true_idx: int) -> int:
    return 1 + int(np.sum(post > post[true_idx]))


In [5]:
import numpy as np

# ============================================================
# Step 2. Real decoder coefficient matrix H_eff
# ============================================================
# Extracted from 2023 reference code: alpha_ij_pow[30][45]
# shape = (n-k, n-1) = (30, 45) for HQC128

H_eff = np.array([[2, 4, 8, 16, 32, 64, 128, 29, 58, 116, 232, 205, 135, 19, 38, 76, 152, 45, 90, 180, 117, 234, 201, 143, 3, 6, 12, 24, 48, 96, 192, 157, 39, 78, 156, 37, 74, 148, 53, 106, 212, 181, 119, 238, 193], [4, 16, 64, 29, 116, 205, 19, 76, 45, 180, 234, 143, 6, 24, 96, 157, 78, 37, 148, 106, 181, 238, 159, 70, 5, 20, 80, 93, 105, 185, 222, 95, 97, 153, 94, 101, 137, 30, 120, 253, 211, 107, 177, 254, 223], [8, 64, 58, 205, 38, 45, 117, 143, 12, 96, 39, 37, 53, 181, 193, 70, 10, 80, 186, 185, 161, 97, 47, 101, 15, 120, 231, 107, 127, 223, 182, 217, 134, 68, 26, 208, 206, 62, 237, 59, 197, 102, 23, 184, 169], [16, 29, 205, 76, 180, 143, 24, 157, 37, 106, 238, 70, 20, 93, 185, 95, 153, 101, 30, 253, 107, 254, 91, 217, 17, 13, 208, 129, 248, 59, 151, 133, 184, 79, 132, 168, 82, 73, 228, 230, 198, 252, 123, 227, 150], [32, 116, 38, 180, 3, 96, 156, 106, 193, 5, 160, 185, 190, 94, 15, 253, 214, 223, 226, 17, 26, 103, 124, 59, 51, 46, 169, 132, 77, 85, 114, 230, 145, 215, 255, 150, 55, 174, 100, 28, 167, 89, 239, 172, 36], [64, 205, 45, 143, 96, 37, 181, 70, 80, 185, 97, 101, 120, 107, 223, 217, 68, 208, 62, 59, 102, 184, 33, 168, 85, 228, 191, 252, 241, 150, 110, 130, 7, 221, 89, 195, 138, 61, 251, 44, 207, 173, 8, 58, 38], [128, 19, 117, 24, 156, 181, 140, 93, 161, 94, 60, 107, 163, 67, 26, 129, 147, 102, 109, 132, 41, 57, 209, 252, 255, 98, 87, 200, 224, 89, 155, 18, 245, 11, 233, 173, 16, 232, 45, 3, 157, 53, 159, 40, 185], [29, 76, 143, 157, 106, 70, 93, 95, 101, 253, 254, 217, 13, 129, 59, 133, 79, 168, 73, 230, 252, 227, 149, 130, 28, 81, 195, 18, 247, 44, 27, 2, 58, 152, 3, 39, 212, 140, 186, 190, 202, 231, 225, 175, 26], [58, 45, 12, 37, 193, 80, 161, 101, 231, 223, 134, 208, 237, 102, 169, 168, 146, 191, 179, 150, 87, 7, 166, 195, 36, 251, 125, 173, 64, 38, 143, 39, 181, 10, 185, 47, 120, 127, 217, 26, 62, 197, 184, 21, 85], [116, 180, 96, 106, 5, 185, 94, 253, 223, 17, 103, 59, 46, 132, 85, 230, 215, 150, 174, 28, 89, 172, 244, 44, 108, 32, 38, 3, 156, 193, 160, 190, 15, 214, 226, 26, 124, 51, 169, 77, 114, 145, 255, 55, 100], [232, 234, 39, 238, 160, 97, 60, 254, 134, 103, 118, 184, 84, 57, 145, 227, 220, 7, 162, 172, 245, 176, 71, 58, 180, 192, 181, 40, 95, 15, 177, 175, 208, 147, 46, 21, 73, 99, 241, 55, 200, 166, 43, 122, 44], [205, 143, 37, 70, 185, 101, 107, 217, 208, 59, 184, 168, 228, 252, 150, 130, 221, 195, 61, 44, 173, 58, 117, 39, 193, 186, 47, 231, 182, 26, 237, 23, 21, 146, 145, 219, 87, 56, 242, 36, 139, 54, 64, 45, 96], [135, 6, 53, 20, 190, 120, 163, 13, 237, 46, 84, 228, 229, 98, 100, 81, 69, 251, 131, 32, 45, 192, 238, 186, 94, 187, 217, 189, 236, 169, 82, 209, 241, 220, 28, 242, 72, 22, 173, 116, 201, 37, 140, 222, 15], [19, 24, 181, 93, 94, 107, 67, 129, 102, 132, 57, 252, 98, 200, 89, 18, 11, 173, 232, 3, 53, 40, 194, 231, 226, 189, 197, 158, 170, 145, 75, 25, 166, 69, 235, 54, 29, 234, 37, 5, 95, 120, 91, 52, 59], [38, 96, 193, 185, 15, 223, 26, 59, 169, 85, 145, 150, 100, 89, 36, 44, 1, 38, 96, 193, 185, 15, 223, 26, 59, 169, 85, 145, 150, 100, 89, 36, 44, 1, 38, 96, 193, 185, 15, 223, 26, 59, 169, 85, 145], [76, 157, 70, 95, 253, 217, 129, 133, 168, 230, 227, 130, 81, 18, 44, 2, 152, 39, 140, 190, 231, 175, 31, 23, 77, 209, 219, 25, 162, 36, 88, 4, 45, 78, 5, 97, 211, 67, 62, 46, 154, 191, 171, 50, 89], [152, 78, 10, 153, 214, 68, 147, 79, 146, 215, 220, 221, 69, 11, 1, 152, 78, 10, 153, 214, 68, 147, 79, 146, 215, 220, 221, 69, 11, 1, 152, 78, 10, 153, 214, 68, 147, 79, 146, 215, 220, 221, 69, 11, 1], [45, 37, 80, 101, 223, 208, 102, 168, 191, 150, 7, 195, 251, 173, 38, 39, 10, 47, 127, 26, 197, 21, 115, 219, 100, 242, 245, 54, 205, 96, 70, 97, 107, 68, 59, 33, 228, 241, 130, 89, 61, 207, 58, 12, 193], [90, 148, 186, 30, 226, 62, 109, 73, 179, 174, 162, 61, 131, 232, 96, 140, 153, 127, 52, 51, 168, 99, 98, 56, 172, 22, 8, 234, 212, 185, 240, 67, 237, 79, 114, 241, 25, 121, 245, 108, 19, 39, 20, 188, 223], [180, 106, 185, 253, 17, 59, 132, 230, 150, 28, 172, 44, 32, 3, 193, 190, 214, 26, 51, 77, 145, 55, 167, 36, 233, 116, 96, 5, 94, 223, 103, 46, 85, 215, 174, 89, 244, 108, 38, 156, 160, 15, 226, 124, 169], [117, 181, 161, 107, 26, 102, 41, 252, 87, 89, 245, 173, 45, 53, 185, 231, 68, 197, 168, 145, 110, 166, 61, 54, 38, 37, 186, 120, 134, 59, 21, 191, 196, 221, 36, 207, 205, 39, 80, 15, 217, 237, 33, 115, 150], [234, 238, 97, 254, 103, 184, 57, 227, 7, 172, 176, 58, 192, 40, 15, 175, 147, 21, 99, 55, 166, 122, 216, 45, 106, 222, 107, 52, 133, 85, 123, 50, 195, 11, 32, 12, 140, 188, 182, 124, 158, 115, 49, 224, 36], [201, 159, 47, 91, 124, 33, 209, 149, 166, 244, 71, 117, 238, 194, 223, 31, 79, 115, 98, 167, 61, 216, 90, 181, 190, 254, 206, 218, 213, 150, 224, 72, 54, 152, 106, 161, 177, 189, 184, 114, 171, 56, 18, 131, 38], [143, 70, 101, 217, 59, 168, 252, 130, 195, 44, 58, 39, 186, 231, 26, 23, 146, 219, 56, 36, 54, 45, 181, 97, 223, 62, 33, 191, 110, 89, 251, 8, 12, 10, 15, 134, 197, 41, 179, 100, 86, 125, 205, 37, 185], [3, 5, 15, 17, 51, 85, 255, 28, 36, 108, 180, 193, 94, 226, 59, 77, 215, 100, 172, 233, 38, 106, 190, 223, 124, 132, 145, 174, 239, 44, 116, 156, 185, 214, 103, 169, 230, 55, 89, 235, 32, 96, 160, 253, 26], [6, 20, 120, 13, 46, 228, 98, 81, 251, 32, 192, 186, 187, 189, 169, 209, 220, 242, 22, 116, 37, 222, 254, 62, 132, 63, 130, 43, 250, 38, 212, 194, 182, 147, 77, 179, 141, 9, 54, 180, 159, 101, 67, 151, 85], [12, 80, 231, 208, 169, 191, 87, 195, 125, 38, 181, 47, 217, 197, 85, 219, 221, 245, 8, 96, 186, 107, 206, 33, 145, 130, 86, 207, 45, 193, 101, 134, 102, 146, 150, 166, 251, 64, 39, 185, 127, 62, 21, 252, 100], [24, 93, 107, 129, 132, 252, 200, 18, 173, 3, 40, 231, 189, 158, 145, 25, 69, 54, 234, 5, 120, 52, 218, 191, 174, 43, 207, 90, 35, 15, 136, 92, 115, 220, 239, 125, 76, 238, 101, 17, 133, 228, 149, 121, 44], [48, 105, 127, 248, 77, 241, 224, 247, 64, 156, 95, 182, 236, 170, 150, 162, 11, 205, 212, 94, 134, 133, 213, 110, 239, 250, 45, 35, 30, 26, 218, 99, 130, 69, 108, 143, 40, 211, 206, 132, 229, 7, 144, 2, 96], [96, 185, 223, 59, 85, 150, 89, 44, 38, 193, 15, 26, 169, 145, 100, 36, 1, 96, 185, 223, 59, 85, 150, 89, 44, 38, 193, 15, 26, 169, 145, 100, 36, 1, 96, 185, 223, 59, 85, 150, 89, 44, 38, 193, 15]], dtype=np.uint16)

print("H_eff.shape =", H_eff.shape)

def get_h_list_for_byte_from_H(H_eff: np.ndarray, byte_idx: int):
    """
    H_eff shape: (m, n_bytes)
    return: coefficient list for one secret byte (one decoder window)
    """
    if H_eff.ndim != 2:
        raise ValueError(f"H_eff must be 2D, got shape={H_eff.shape}")
    m, n_bytes = H_eff.shape
    if not (0 <= byte_idx < n_bytes):
        raise IndexError(f"byte_idx={byte_idx} out of range for n_bytes={n_bytes}")
    return np.array(H_eff[:, byte_idx], dtype=np.uint16)

def run_one_byte_with_real_H(H_eff: np.ndarray, byte_idx: int, rng):
    """
    기존 aggregate_scores_for_one_true_byte(...)를 그대로 사용하되,
    랜덤 h_list 대신 실제 H_eff[:, byte_idx]를 사용한다.
    """
    h_list = get_h_list_for_byte_from_H(H_eff, byte_idx)
    c_true = int(rng.integers(0, 256))
    res = aggregate_scores_for_one_true_byte(c_true, h_list, rng)
    return {
        "byte_idx": int(byte_idx),
        "c_true": int(c_true),
        "h_list": h_list,
        "res": res,
    }


H_eff.shape = (30, 45)


In [6]:
# ----------------------------
# Occurrence sampler using Gaussian posterior calibrated to Table 2 accuracies
# ----------------------------
def sample_one_occurrence_for_true_c_accuracy_only(c_true: int, h: int, rng):
    hw_c = hw8_scalar(c_true)
    y_true = gf_mul_ref_python(h, c_true)
    hw_y = hw8_scalar(y_true)

    # op1 evidence: HW(c)
    p_op1 = gaussian_hw_posterior(
        true_hw=hw_c,
        alpha=ALPHA_OP1,
        beta=BETA_OP1,
        sigma_raw=SIGMA_RAW_OP1,
        rng=rng,
    )

    # output evidence: HW(gf_mul(h,c))
    p_out = gaussian_hw_posterior(
        true_hw=hw_y,
        alpha=ALPHA_OUT,
        beta=BETA_OUT,
        sigma_raw=SIGMA_RAW_OUT,
        rng=rng,
    )

    return {
        "c_true": int(c_true),
        "h": int(h),
        "y_true": int(y_true),
        "hw_c": int(hw_c),
        "hw_y": int(hw_y),
        "p_op1": p_op1,
        "p_out": p_out,
    }

def get_h_list_for_byte_from_H(H_eff: np.ndarray, byte_idx: int):
    return np.array(H_eff[:, byte_idx], dtype=np.uint16)

def aggregate_scores_for_one_true_byte_accuracy_only(c_true: int, h_list, rng):
    log_scores = np.zeros(256, dtype=np.float64)
    occ_records = []
    for h in h_list:
        occ = sample_one_occurrence_for_true_c_accuracy_only(c_true, int(h), rng)
        occ_records.append(occ)
        for c in range(256):
            hw_c = hw8_scalar(c)
            y = gf_mul_ref_python(int(h), c)
            hw_y = hw8_scalar(y)
            p1 = occ["p_op1"][hw_c]
            p2 = occ["p_out"][hw_y]
            log_scores[c] += np.log(max(p1, 1e-300)) + np.log(max(p2, 1e-300))
    m = np.max(log_scores)
    post = np.exp(log_scores - m)
    post /= post.sum()
    return {
        "post": post,
        "best_c": int(np.argmax(post)),
        "rank_true": posterior_rank(post, c_true),
        "occ_records": occ_records,
    }

def run_one_byte_with_real_H_accuracy_only(H_eff: np.ndarray, byte_idx: int, rng):
    h_list = get_h_list_for_byte_from_H(H_eff, byte_idx)
    c_true = int(rng.integers(0, 256))
    res = aggregate_scores_for_one_true_byte_accuracy_only(c_true, h_list, rng)
    return {"byte_idx": int(byte_idx), "c_true": int(c_true), "h_list": h_list, "res": res}


In [7]:
# ----------------------------
# Figure 5 BP using accuracy-only evidence
# ----------------------------
NC_FIG5 = 512
# log(0) sentinel must be outside valid log-sum range [0, 508]
GF_SENTINEL = 511
GF_ALPHA = 2

def build_hqc_logexp_tables():
    exp_raw = np.zeros(255, dtype=np.uint32)
    log_raw = np.full(256, GF_SENTINEL, dtype=np.uint32)
    x = 1
    for i in range(255):
        exp_raw[i] = x
        log_raw[x] = i
        x = gf_mul_ref_python(GF_ALPHA, x)
    assert x == 1

    hw512 = np.zeros(NC_FIG5, dtype=np.uint32)
    hw512[:256] = np.array([hw8_scalar(v) for v in range(256)], dtype=np.uint32)

    logz512 = np.zeros(NC_FIG5, dtype=np.uint32)
    logz512[:256] = log_raw

    mod255z = np.zeros(NC_FIG5, dtype=np.uint32)
    for v in range(NC_FIG5):
        if v == GF_SENTINEL:
            mod255z[v] = GF_SENTINEL
        elif v <= 508:
            mod255z[v] = v % 255
        else:
            mod255z[v] = GF_SENTINEL

    expz512 = np.zeros(NC_FIG5, dtype=np.uint32)
    for i in range(255):
        expz512[i] = exp_raw[i]
    expz512[GF_SENTINEL] = 0

    return hw512, logz512, mod255z, expz512, log_raw, exp_raw

def hw_posterior_to_512state(p_hw):
    evid = np.zeros(NC_FIG5, dtype=np.float64)
    evid[:9] = p_hw.astype(np.float64)
    evid /= evid.sum()
    return evid

def byte_uniform_support_512():
    evid = np.zeros(NC_FIG5, dtype=np.float64)
    evid[:256] = 1.0 / 256.0
    return evid

def build_addh_table(h, log_raw):
    h_log = int(log_raw[int(h)])
    addh = np.zeros(NC_FIG5, dtype=np.uint32)
    for x in range(NC_FIG5):
        if x == GF_SENTINEL:
            addh[x] = GF_SENTINEL
        elif x < 255:
            addh[x] = x + h_log
        else:
            addh[x] = GF_SENTINEL
    return addh

def get_distribution_1d(dist):
    dist = np.asarray(dist, dtype=np.float64)
    if dist.ndim == 2:
        dist = dist[0]
    return dist

hw512, logz512, mod255z, expz512, log_raw, exp_raw = build_hqc_logexp_tables()

def run_figure5_window_bp_accuracy_only(H_eff, byte_idx, rng):
    trial = run_one_byte_with_real_H_accuracy_only(H_eff, byte_idx=byte_idx, rng=rng)
    c_true = trial["c_true"]
    h_list = trial["h_list"]
    occ_records = trial["res"]["occ_records"]

    tables = {"hw": hw512, "logz": logz512, "mod255z": mod255z, "expz": expz512}
    graph_desc = f"NC {NC_FIG5}\nTABLE hw\nTABLE logz\nTABLE mod255z\nTABLE expz\nVAR SINGLE c\n"

    for i, h in enumerate(h_list):
        addh_name = f"addh{i}"
        lc_name, s_name, r_name = f"lc{i}", f"s{i}", f"r{i}"
        y_name, hc_name, hy_name = f"y{i}", f"hc{i}", f"hy{i}"
        tables[addh_name] = build_addh_table(h, log_raw)
        graph_desc += f"TABLE {addh_name}\n"
        graph_desc += f"VAR SINGLE {lc_name}\nVAR SINGLE {s_name}\nVAR SINGLE {r_name}\n"
        graph_desc += f"VAR SINGLE {y_name}\nVAR SINGLE {hc_name}\nVAR SINGLE {hy_name}\n"
        graph_desc += f"PROPERTY {lc_name} = logz[c]\n"
        graph_desc += f"PROPERTY {s_name}  = {addh_name}[{lc_name}]\n"
        graph_desc += f"PROPERTY {r_name}  = mod255z[{s_name}]\n"
        graph_desc += f"PROPERTY {y_name}  = expz[{r_name}]\n"
        graph_desc += f"PROPERTY {hc_name} = hw[c]\n"
        graph_desc += f"PROPERTY {hy_name} = hw[{y_name}]\n"

    factor_graph = FactorGraph(graph_desc, tables)
    bp = BPState(factor_graph, 1, {})
    bp.set_evidence("c", np.ascontiguousarray(byte_uniform_support_512(), dtype=np.float64))
    for i, occ in enumerate(occ_records):
        bp.set_evidence(f"hc{i}", np.ascontiguousarray(hw_posterior_to_512state(occ["p_op1"]), dtype=np.float64))
        bp.set_evidence(f"hy{i}", np.ascontiguousarray(hw_posterior_to_512state(occ["p_out"]), dtype=np.float64))
    bp.bp_acyclic("c")
    bp_dist_512 = get_distribution_1d(bp.get_distribution("c"))
    bp_dist_512 /= bp_dist_512.sum()
    bp_post = bp_dist_512[:256].copy()
    bp_post /= bp_post.sum()
    return {
        "byte_idx": int(byte_idx),
        "c_true": int(c_true),
        "bp_post": bp_post,
        "bp_best": int(np.argmax(bp_post)),
        "bp_rank_true": posterior_rank(bp_post, c_true),
    }


In [8]:
# ----------------------------
# Accuracy-only evaluation (Section 4.2 style)
# ----------------------------
TAU = 19  # HQC128

def run_full_window_attack_once_accuracy_only(H_eff, rng):
    all_res = [run_figure5_window_bp_accuracy_only(H_eff, byte_idx=j, rng=rng) for j in range(H_eff.shape[1])]
    byte_true = np.array([res["c_true"] for res in all_res], dtype=np.uint16)
    byte_best = np.array([res["bp_best"] for res in all_res], dtype=np.uint16)
    byte_ranks = np.array([res["bp_rank_true"] for res in all_res], dtype=np.int32)
    n_errors = int(np.sum(byte_best != byte_true))
    redec_success = (n_errors <= (TAU - 1))  # C1 consumes one slot
    return {
        "byte_true": byte_true,
        "byte_best": byte_best,
        "byte_ranks": byte_ranks,
        "n_errors": n_errors,
        "redec_success": bool(redec_success),
    }

def monte_carlo_redecoding_accuracy_only(H_eff, n_attacks=30, seed=2026):
    rng_local = np.random.default_rng(seed)
    success = 0
    err_counts = []
    top1_list = []
    mean_rank_list = []
    for _ in range(n_attacks):
        r = run_full_window_attack_once_accuracy_only(H_eff, rng=rng_local)
        success += int(r["redec_success"])
        err_counts.append(r["n_errors"])
        top1_list.append(np.mean(r["byte_best"] == r["byte_true"]))
        mean_rank_list.append(np.mean(r["byte_ranks"]))
    return {
        "success_rate_redec": success / n_attacks,
        "avg_errors": float(np.mean(err_counts)),
        "max_errors": int(np.max(err_counts)),
        "avg_top1": float(np.mean(top1_list)),
        "avg_mean_rank": float(np.mean(mean_rank_list)),
        "err_counts": np.array(err_counts, dtype=np.int32),
    }

# Monte Carlo run with fixed Table 2 accuracies
mc_redec = monte_carlo_redecoding_accuracy_only(H_eff, n_attacks=N_ATTACKS, seed=2026)
print("[Gaussian calibrated Table 2 Monte Carlo re-decoding evaluation]")
print("ACC_OP1_HW         =", ACC_OP1_HW)
print("ACC_OUT_HW         =", ACC_OUT_HW)
print("ALPHA_OP1          =", ALPHA_OP1)
print("ALPHA_OUT          =", ALPHA_OUT)
print("SIGMA_RAW_OP1      =", SIGMA_RAW_OP1)
print("SIGMA_RAW_OUT      =", SIGMA_RAW_OUT)
print("SIGMA_EFF_OP1      =", SIGMA_EFF_OP1)
print("SIGMA_EFF_OUT      =", SIGMA_EFF_OUT)
print("n_attacks          =", N_ATTACKS)
print("success_rate_redec =", mc_redec["success_rate_redec"])
print("avg_errors         =", mc_redec["avg_errors"])
print("max_errors         =", mc_redec["max_errors"])
print("avg_top1           =", mc_redec["avg_top1"])
print("avg_mean_rank      =", mc_redec["avg_mean_rank"])


[Gaussian calibrated Table 2 Monte Carlo re-decoding evaluation]
ACC_OP1_HW         = 0.3035
ACC_OUT_HW         = 0.5178
ALPHA_OP1          = 1.0
ALPHA_OUT          = 4.0
SIGMA_RAW_OP1      = 1.2941357193168332
SIGMA_RAW_OUT      = 2.8582129061098205
SIGMA_EFF_OP1      = 1.2941357193168332
SIGMA_EFF_OUT      = 0.7145532265274551
n_attacks          = 30
success_rate_redec = 1.0
avg_errors         = 0.16666666666666666
max_errors         = 1
avg_top1           = 0.9962962962962963
avg_mean_rank      = 1.0007407407407407
